### CARGA DE LIBRERÍAS NECESARIAS PARA LA LIMPIEZA


In [1]:
import pandas as pd
import numpy as np
import os

### CARGA DEL ARCHIVO A LIMPIAR

In [2]:
Ruta_Archivo_Entrada = '../Original_Data/2025_3T_Flujospororigen_actu__4_.xlsx'
Ruta_Archivo_Salida = '../Prepared_Data/Country'
Archivo_Excel = pd.read_excel(Ruta_Archivo_Entrada, sheet_name = None, header=None)

In [3]:
'''Ubicación del entorno de trabajo y hojas de excel contenidas dentro del archivo'''
#print(os.getcwd())
Archivo_Excel.keys()

dict_keys(['Por tipo de inversión', 'Por entidad federativa', 'Por sector'])

### LIMPIEZA DE LA PRIMERA HOJA DEL ARCHIVO: ''Por tipo de inversión'

##### Selección del archivo a limpiar 

In [4]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Investment_Types_by_Country = Archivo_Excel['Por tipo de inversión']
#display(Investment_Types_by_Country)
Investment_Types_by_Country_Description = Investment_Types_by_Country.iloc[0,0]
#print(Investment_Types_by_Country_Description)
Investment_Types_by_Country_Database = Investment_Types_by_Country.iloc[-1,0]
#print(Investment_Types_by_Country_DataBase)

##### Limpieza general de dataframe

In [5]:
''' Se elimninan las dos primeras filas y las tres últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Investment_Types_by_Country_1 = Investment_Types_by_Country.copy()
Investment_Types_by_Country_1 = Investment_Types_by_Country_1.iloc[2:,:].reset_index(drop=True)
Investment_Types_by_Country_1 = Investment_Types_by_Country_1.iloc[:-3,:].reset_index(drop=True)
#display(Investment_Types_by_Country_1)
'''
Dado el formato de origen en excel y el hecho de que algunas celdas estan combinadas, al cargar el archivo los valores contenidos en varias celdas no se han distribuido de
manera uniforme en las celdas de la misma fila en las que se contenia, por lo que al contenerse dentro de la celda más a la izquierda, se rellena hacía la derecha, esto 
nos devuelve las columnas con los años, donde antes se contenian y por último se combinan la celdas de los años y trimestres para obtener fechas completas.
'''
''' Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Investment_Types_by_Country_1.iloc[0,1:] = Investment_Types_by_Country_1.iloc[0,1:].ffill()
''' Se extraen los años contenidos en la fila 0 '''
Investment_Types_by_Country_1_Años = Investment_Types_by_Country_1.iloc[0,1:].astype(int).tolist()
#print(Investment_Types_by_Country_1_Años)
''' Se extraen los trimestres contenidos en la fila 1 '''
Investment_Types_by_Country_1_Trimestres = Investment_Types_by_Country_1.iloc[1,1:].astype(int).tolist()
#print(Investment_Types_by_Country_1_Trimestres)
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo en formato adecuado '''
Investment_Types_by_Country_1_Fecha = [f"{a}Q{t}" for a, t in zip(Investment_Types_by_Country_1_Años, Investment_Types_by_Country_1_Trimestres)]
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Investment_Types_by_Country_1.columns = [Investment_Types_by_Country_1.iloc[0,0]] + Investment_Types_by_Country_1_Fecha
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados'''
Investment_Types_by_Country_1 = Investment_Types_by_Country_1.iloc[2:,:].reset_index(drop=True)
#display(Investment_Types_by_Country_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_31132\3702310313.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Investment_Types_by_Country_1.iloc[0,1:] = Investment_Types_by_Country_1.iloc[0,1:].ffill()


##### Ingeniería de datos, métodos de categorización

In [6]:
'''
En esta sección aplicara ingeniería de datos, se categorizará por etiquetas basandonos en el nombre de los países inversores.
Se categorizará por etiqueta de países, por lo tanto, se creará el diccionario con el nombre de dichos países extrayendolos directamente, 
esto será útil para representar información a través de un formato largo y series temporales basandonos en categorías.
'''
'''Se genera la copia del archivo con el cual trabajaremos'''
Investment_Types_by_Country_2 = Investment_Types_by_Country_1.copy()
'''A continuación se creará una nueva columna, esta contendra a los indicadores por país'''
Investment_Types_by_Country_2['País'] = ''
''' Rellenamos el valor de la columna País para la fila de Total general con ese mismo valor '''
Investment_Types_by_Country_2.loc[0,'País'] = 'Total general'
'''Antes de categorizar por país, primero tenemos que obtener todos los estados y conceptos contenidos dentro de nuestro dataframe'''
Countries_Concepts = Investment_Types_by_Country_2['Origen / Tipo de Inversión '].unique()
#print(Countries_Concepts)
'''Conociendo los conceptos que se encuentran en esta hoja de excel en particular, procedemos a crear una lista con ellos'''
Concepts = ['Total general','Nuevas inversiones','Cuentas entre compañías','Reinversión de utilidades']
'''Creamos la lista que necesitamos de países en nuestro dataframe únicamente usando la función lambda y filter, extraemos la información que necesitamos'''
Country = list(filter(lambda x: x not in Concepts, Countries_Concepts))
#print(Country)
''' Una vez hecho lo anterior, copiamos el valor de la columna "Origen / Tipo de Inversión" si está en la lista de países creada anteriormente '''
Investment_Types_by_Country_2.loc[Investment_Types_by_Country_2['Origen / Tipo de Inversión '].isin(Country),'País'] = Investment_Types_by_Country_2['Origen / Tipo de Inversión ']
''' Reemplazamos los valores vacíos "" por valores nan en la columna "País" '''
Investment_Types_by_Country_2['País'] = Investment_Types_by_Country_2['País'].replace(['',False],np.nan)
''' Ahora rellenamos hacía abajo los valores correspondientes a países en la columna "País" '''
Investment_Types_by_Country_2['País'] = Investment_Types_by_Country_2['País'].ffill(axis=0)
''' A continuación en la columna "Origen / Tipo de Inversión" se sustituyen las celdas con el nombre de algún país puesto que,ya se ha agregado el identificador necesario'''
Investment_Types_by_Country_2.loc[Investment_Types_by_Country_2['Origen / Tipo de Inversión '].isin(Country),'Origen / Tipo de Inversión '] = 'Total general por país'
'''Por último, se renombra la columna de "Origen / Tipo de Inversión" por "Tipo de Inversión" para mayor legibilidad'''
Investment_Types_by_Country_2 = Investment_Types_by_Country_2.rename(columns={'Origen / Tipo de Inversión ':'Tipo de Inversión'})
#display(Investment_Types_by_Country_2)

##### Transformación de dataframe

In [7]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal además de un análisis categórico relativamente sencillo'''
Investment_Types_by_Country_3 = pd.melt(
    Investment_Types_by_Country_2,
    id_vars = ['Tipo de Inversión', 'País'],
    var_name = 'Año_Trimestre',
    value_name = 'IED'
)

'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Investment_Types_by_Country_3['Fecha'] = pd.PeriodIndex(Investment_Types_by_Country_3['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Investment_Types_by_Country_3 = Investment_Types_by_Country_3.drop(['Año_Trimestre'],axis=1)
display(Investment_Types_by_Country_3)

,Tipo de Inversión,País,IED,Fecha
0,Total general,Total general,7437.856436,2006-03-31
1,Total general por país,Afganistán,0,2006-03-31
2,Nuevas inversiones,Afganistán,0,2006-03-31
3,Total general por país,Albania,0,2006-03-31
4,Nuevas inversiones,Albania,0,2006-03-31
...,...,...,...,...
35017,Nuevas inversiones,Zimbabue,0,2025-06-30
35018,Total general por país,Otro origen,1551.61853,2025-06-30
35019,Cuentas entre compañías,Otro origen,-228.948877,2025-06-30
35020,Nuevas inversiones,Otro origen,26.670146,2025-06-30


##### Formato general del archivo

In [8]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Investment_Types_by_Country_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35022 entries, 0 to 35021
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Tipo de Inversión  35022 non-null  object        
 1   País               35022 non-null  object        
 2   IED                35022 non-null  object        
 3   Fecha              35022 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 1.1+ MB


##### Carga de archivo al entorno local

In [9]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_1 = 'Investment_Types_by_Country.csv'
Ruta_Salida_1 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_1)
Investment_Types_by_Country_3.to_csv(Ruta_Salida_1, index=False, encoding='utf-8-sig')

### LIMPIEZA DE LA SEGUNDA HOJA DEL ARCHIVO: 'Por entidad federativa'

##### Selección del archivo a limpiar 

In [10]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Country_Origin_Investment_by_Country = Archivo_Excel['Por entidad federativa']
#display(Country_Origin_Investment_by_Country)
Country_Origin_Investment_by_Country_Description = Country_Origin_Investment_by_Country.iloc[0,0]
#print(Country_Origin_Investment_by_Country_Description)
Country_Origin_Investment_by_Country_Database = Country_Origin_Investment_by_Country.iloc[-1,0]
#print(Country_Origin_Investment_by_Country_Database)

##### Limpieza general de dataframe

In [11]:
''' Se elimninan las dos primeras filas y las tres últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Country_Origin_Investment_by_Country_1 = Country_Origin_Investment_by_Country.copy()
Country_Origin_Investment_by_Country_1 = Country_Origin_Investment_by_Country_1.iloc[2:,:].reset_index(drop=True)
Country_Origin_Investment_by_Country_1 = Country_Origin_Investment_by_Country_1.iloc[:-4,:].reset_index(drop=True)
'''
Dado el formato de origen en excel y el hecho de que algunas celdas estan combinadas, al cargar el archivo los valores contenidos en varias celdas no se han distribuido de
manera uniforme en las celdas de la misma fila en las que se contenia, por lo que al contenerse dentro de la celda más a la izquierda, se rellena hacía la derecha, esto 
nos devuelve las columnas con los años, donde antes se contenian y por último se combinan la celdas de los años y trimestres para obtener fechas completas.
'''
''' Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Country_Origin_Investment_by_Country_1.iloc[0,1:] = Country_Origin_Investment_by_Country_1.iloc[0,1:].ffill()
''' Se extraen los años contenidos en la fila 0 '''
Country_Origin_Investment_by_Country_1_Años = Country_Origin_Investment_by_Country_1.iloc[0,1:].astype(int).tolist()
#print(Country_Origin_Investment_by_Country_1_Años)
''' Se extraen los trimestres contenidos en la fila 1 '''
Country_Origin_Investment_by_Country_1_Trimestres = Country_Origin_Investment_by_Country_1.iloc[1,1:].astype(int).tolist()
#print(Country_Origin_Investment_by_Country_1_Trimestres)
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo en formato adecuado '''
Country_Origin_Investment_by_Country_1_Fecha = [f"{a}Q{t}" for a, t in zip(Country_Origin_Investment_by_Country_1_Años,Country_Origin_Investment_by_Country_1_Trimestres)]
#print(Country_Origin_Investment_by_Country_1_Fecha)
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Country_Origin_Investment_by_Country_1.columns = [Country_Origin_Investment_by_Country_1.iloc[0,0]] + Country_Origin_Investment_by_Country_1_Fecha
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados'''
Country_Origin_Investment_by_Country_1 = Country_Origin_Investment_by_Country_1.iloc[2:,:].reset_index(drop=True)
#display(Country_Origin_Investment_by_Country_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_31132\3278011303.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Country_Origin_Investment_by_Country_1.iloc[0,1:] = Country_Origin_Investment_by_Country_1.iloc[0,1:].ffill()


##### Ingeniería de datos, métodos de categorización

In [12]:
'''
En esta sección aplicara ingeniería de datos, se categorizará por etiquetas basandonos en el nombre de los países inversores.
Se categorizará por etiqueta de países, por lo tanto, se creará el diccionario con el nombre de dichos países extrayendolos directamente, 
esto será útil para representar información a través de un formato largo y series temporales basandonos en categorías.
'''
'''Se genera la copia del archivo con el cual trabajaremos'''
Country_Origin_Investment_by_Country_2 = Country_Origin_Investment_by_Country_1.copy()
'''A continuación se creará una nueva columna, esta contendra a los indicadores por país'''
Country_Origin_Investment_by_Country_2['País'] = ''
''' Rellenamos el valor de la columna País para la fila de Total general con ese mismo valor '''
Country_Origin_Investment_by_Country_2.loc[0,'País'] = 'Total general'
'''Antes de categorizar por país, primero tenemos que obtener todos los países y estados contenidos dentro de nuestro dataframe'''
Countries_States = Country_Origin_Investment_by_Country_2['Origen / Entidad Federativa'].unique()
#print(Countries_States)
'''Conociendo los estados que se encuentran en esta hoja de excel en particular, procedemos a crear una lista con ellos'''
States = ["Total general","Aguascalientes", "Baja California", "Baja California Sur", "Campeche", "Chiapas", "Chihuahua", "Ciudad de México", "Coahuila de Zaragoza",
           "Colima", "Durango", "Estado de México", "Guanajuato", "Guerrero","Hidalgo", "Jalisco", "Michoacán de Ocampo", "Morelos", "Nayarit",
           "Nuevo León", "Oaxaca", "Puebla", "Querétaro", "Quintana Roo", "San Luis Potosí", "Sinaloa", "Sonora", "Tabasco", "Tamaulipas",
           "Tlaxcala", "Veracruz de Ignacio de la Llave", "Yucatán", "Zacatecas"]
#print(States)
'''Creamos la lista que necesitamos de países en nuestro dataframe únicamente usando la función lambda y filter, extraemos la información que necesitamos'''
Country = list(filter(lambda x: x not in States, Countries_States ))
#print(Country)
''' Una vez hecho lo anterior, copiamos el valor de la columna "Origen / Tipo de Inversión" si está en la lista de países creada anteriormente '''
Country_Origin_Investment_by_Country_2.loc[Country_Origin_Investment_by_Country_2['Origen / Entidad Federativa'].isin(Country),'País'] = Country_Origin_Investment_by_Country_2['Origen / Entidad Federativa']
''' Reemplazamos los valores vacíos "" por valores nan en la columna "País" '''
Country_Origin_Investment_by_Country_2['País'] = Country_Origin_Investment_by_Country_2['País'].replace(['',False],np.nan)
''' Ahora rellenamos hacía abajo los valores correspondientes a países en la columna "País" '''
Country_Origin_Investment_by_Country_2['País'] = Country_Origin_Investment_by_Country_2['País'].ffill(axis=0)
''' A continuación en la columna "Origen / Entidad Federativa" se sustituyen las celdas con el nombre de algún país puesto que,ya se ha agregado el identificador necesario'''
Country_Origin_Investment_by_Country_2.loc[Country_Origin_Investment_by_Country_2['Origen / Entidad Federativa'].isin(Country),'Origen / Entidad Federativa'] = 'Total general por país'
'''Por último, se renombra la columna de "Origen / Tipo de Inversión" por "Tipo de Inversión" para mayor legibilidad'''
Country_Origin_Investment_by_Country_2 = Country_Origin_Investment_by_Country_2.rename(columns = {'Origen / Entidad Federativa':'Inversión en cada estado por país'}).reset_index(drop=True)
#display(Country_Origin_Investment_by_Country_2)

##### Transformación de dataframe

In [13]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal además de un análisis categórico relativamente sencillo'''
Country_Origin_Investment_by_Country_3 = pd.melt(
    Country_Origin_Investment_by_Country_2,
    id_vars = ['Inversión en cada estado por país','País'],
    var_name= 'Año_Trimestre',
    value_name = 'IED'
)

'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Country_Origin_Investment_by_Country_3['Fecha'] = pd.PeriodIndex(Country_Origin_Investment_by_Country_3['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Country_Origin_Investment_by_Country_3 = Country_Origin_Investment_by_Country_3.drop(['Año_Trimestre'], axis=1)
display(Country_Origin_Investment_by_Country_3)

,Inversión en cada estado por país,País,IED,Fecha
0,Total general,Total general,7437.856436,2006-03-31
1,Total general por país,Afganistán,0,2006-03-31
2,Yucatán,Afganistán,0,2006-03-31
3,Total general por país,Albania,0,2006-03-31
4,Quintana Roo,Albania,0,2006-03-31
...,...,...,...,...
165433,Chiapas,Otro origen,C,2025-06-30
165434,Nayarit,Otro origen,-0.121044,2025-06-30
165435,Baja California Sur,Otro origen,0.004897,2025-06-30
165436,Puebla,Otro origen,0,2025-06-30


##### Formato general del archivo

In [14]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Country_Origin_Investment_by_Country_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165438 entries, 0 to 165437
Data columns (total 4 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   Inversión en cada estado por país  165438 non-null  object        
 1   País                               165438 non-null  object        
 2   IED                                165438 non-null  object        
 3   Fecha                              165438 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 5.0+ MB


##### Carga de datos al entorno local

In [15]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_2 = 'Investment_by_State_from_the_Countries.csv'
Ruta_Salida_2 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_2)
Country_Origin_Investment_by_Country_3.to_csv(Ruta_Salida_2, index=False, encoding='utf-8-sig')

### LIMPIEZA DE LA TERCERA HOJA DEL ARCHIVO: 'Por sector'

##### Selección del archivo a limpiar 

In [16]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Investment_by_Sector_from_the_Countries = Archivo_Excel['Por sector']
#display(Investment_by_Sector_from_the_Countries)
Investment_by_Sector_from_the_Countries_Description = Investment_by_Sector_from_the_Countries.iloc[0,0]
#print(Investment_by_Sector_from_the_Countries_Description)
Investment_by_Sector_from_the_Countries_Database = Investment_by_Sector_from_the_Countries.iloc[-1,0]
#print(Investment_by_Sector_from_the_Countries_Database)

##### Limpieza general de dataframe

In [17]:
''' Se crea una copia del archivo con el cual trabajaremos '''
Investment_by_Sector_from_the_Countries_1 = Investment_by_Sector_from_the_Countries.copy()
''' Se elimninan las dos primeras filas y las cuatro últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Investment_by_Sector_from_the_Countries_1 = Investment_by_Sector_from_the_Countries_1.iloc[2:,:].reset_index(drop=True)
Investment_by_Sector_from_the_Countries_1 = Investment_by_Sector_from_the_Countries_1.iloc[:-4,:].reset_index(drop=True)
'''
Dado el formato de origen en excel y el hecho de que algunas celdas estan combinadas, al cargar el archivo los valores contenidos en varias celdas no se han distribuido de
manera uniforme en las celdas de la misma fila en las que se contenia, por lo que al contenerse dentro de la celda más a la izquierda, se rellena hacía la derecha, esto 
nos devuelve las columnas con los años, donde antes se contenian y por último se combinan la celdas de los años y trimestres para obtener fechas completas.
'''
''' Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Investment_by_Sector_from_the_Countries_1.iloc[0,1:] = Investment_by_Sector_from_the_Countries_1.iloc[0,1:].ffill()
''' Se extraen los años contenidos en la fila 0 '''
Investment_by_Sector_from_the_Countries_1_Años = Investment_by_Sector_from_the_Countries_1.iloc[0,1:].astype(int).tolist()
#print(Investment_by_Sector_from_the_Countries_1_Años)
''' Se extraen los trimestres contenidos en la fila 1 '''
Investment_by_Sector_from_the_Countries_1_Trimestres = Investment_by_Sector_from_the_Countries_1.iloc[1,1:].astype(int).tolist()
#print(Investment_by_Sector_from_the_Countries_1_Trimestres)
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo con fechas en el formato adecuado '''
Investment_by_Sector_from_the_Countries_1_Fecha = [f"{a}Q{t}" for a,t in zip(Investment_by_Sector_from_the_Countries_1_Años,Investment_by_Sector_from_the_Countries_1_Trimestres)]
#print(Investment_by_Sector_from_the_Countries_1_Fecha)
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Investment_by_Sector_from_the_Countries_1.columns = [Investment_by_Sector_from_the_Countries_1.iloc[0,0]] + Investment_by_Sector_from_the_Countries_1_Fecha
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados '''
Investment_by_Sector_from_the_Countries_1 = Investment_by_Sector_from_the_Countries_1.iloc[2:,:].reset_index(drop=True)
#display(Investment_by_Sector_from_the_Countries_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_31132\866665033.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Investment_by_Sector_from_the_Countries_1.iloc[0,1:] = Investment_by_Sector_from_the_Countries_1.iloc[0,1:].ffill()


##### Ingeniería de datos, métodos de categorización y jerarquización 

In [18]:
'''Se genera la copia del archivo con el cual trabajaremos'''
Investment_by_Sector_from_the_Countries_2 = Investment_by_Sector_from_the_Countries_1.copy()
'''Para jerarquizar y categorizar adecuadamente, primero, se separará la columna "Origen / Actividad Económica" trás el primer espacio, perimitiendo separar de manera correcta el código 
númerico SCIAN y su concepto escrito de manera textual'''
Investment_by_Sector_from_the_Countries_2[['Código','Concepto']] = Investment_by_Sector_from_the_Countries_2['Origen / Actividad Económica'].str.split(r"\s+",n=1,expand=True)
'''A continuación se presenta un método simple para identificar el nivel general de un concepto al igual que su jerarquia dentro de la normativa SCIAN '''
Investment_by_Sector_from_the_Countries_2['Nivel'] = Investment_by_Sector_from_the_Countries_2['Código'].apply(lambda x: 2 if '-' in x else len(x))
Investment_by_Sector_from_the_Countries_2['Sector'] = Investment_by_Sector_from_the_Countries_2['Código'].apply(lambda x: x if '-' in x else x[:2])
Investment_by_Sector_from_the_Countries_2['Subsector'] = Investment_by_Sector_from_the_Countries_2['Código'].apply(lambda x: None if '-' in x or len(x)<3 else x[:3])
Investment_by_Sector_from_the_Countries_2['Rama'] = Investment_by_Sector_from_the_Countries_2['Código'].apply(lambda x: None if '-' in x or len(x)<4 else x[:4])
'''A continuación se establece un método relativamente sencillo para identificar a los países, esto será útil al momento de analizar los datos '''
Investment_by_Sector_from_the_Countries_2.loc[~Investment_by_Sector_from_the_Countries_2['Código'].str.match(r"^\d+$",na=False),'Concepto'] = 'Total general por país'
'''A los datos generales por país se les agrega otro identificador para que sean aún más fácil'''
Investment_by_Sector_from_the_Countries_2.loc[Investment_by_Sector_from_the_Countries_2['Concepto'] == 'Total general por país',['Nivel','Sector','Subsector','Rama']] = 0
'''De la misma manera se establece un identificador para el total'''
Investment_by_Sector_from_the_Countries_2.loc[0,'Concepto'] = 'Total general'
'''Al igua que se agrega un identificador para hacer aún más fácil la aplicación de filtros '''
Investment_by_Sector_from_the_Countries_2.loc[0,'Nivel':] = 1
'''Por último se eliminan y renombrar columnas a conveniencia '''
Investment_by_Sector_from_the_Countries_2 = Investment_by_Sector_from_the_Countries_2.drop(['Código'],axis=1)
Investment_by_Sector_from_the_Countries_2 = Investment_by_Sector_from_the_Countries_2.rename(columns={'Origen / Actividad Económica':'País de origen'})
#display(Investment_by_Sector_from_the_Countries_2)

##### Transformación del dataframe

In [19]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal además de un análisis categórico relativamente sencillo'''
Investment_by_Sector_from_the_Countries_3 = pd.melt(
    Investment_by_Sector_from_the_Countries_2,
    id_vars = ['País de origen','Concepto','Nivel','Sector','Subsector','Rama'],
    var_name = 'Año_Trimestre',
    value_name = 'IED'
) 

'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Investment_by_Sector_from_the_Countries_3['Fecha'] = pd.PeriodIndex(Investment_by_Sector_from_the_Countries_3['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Investment_by_Sector_from_the_Countries_3 = Investment_by_Sector_from_the_Countries_3.drop(['Año_Trimestre'],axis=1)
#display(Investment_by_Sector_from_the_Countries_3)

##### Formato general del archivo

In [20]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Investment_by_Sector_from_the_Countries_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 849420 entries, 0 to 849419
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   País de origen  849420 non-null  object        
 1   Concepto        849420 non-null  object        
 2   Nivel           849420 non-null  int64         
 3   Sector          849420 non-null  object        
 4   Subsector       759486 non-null  object        
 5   Rama            506454 non-null  object        
 6   IED             849420 non-null  object        
 7   Fecha           849420 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 51.8+ MB


##### Carga de datos al entorno local

In [21]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_3 = 'Investment_by_Sector_from_the_Countries.csv'
Ruta_Salida_3 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_3)
Investment_by_Sector_from_the_Countries_3.to_csv(Ruta_Salida_3, index=False, encoding='utf-8-sig')